In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# 1. Generate Synthetic Marketing Time-Series Data
np.random.seed(42)
n_weeks = 156
weeks = pd.date_range(start="2023-01-01", periods=n_weeks, freq="W")

trend = np.linspace(100, 150, n_weeks)
seasonality = 20 * np.sin(2 * np.pi * weeks.strftime('%j').astype(float) / 365.25)
price = np.random.normal(loc=12.0, scale=0.8, size=n_weeks)

spend_tv = np.random.gamma(shape=2, scale=15, size=n_weeks)
spend_digital = np.random.gamma(shape=3, scale=10, size=n_weeks)

df = pd.DataFrame({
    'Date': weeks, 'Base_Trend': trend, 'Seasonality': seasonality,
    'Price': price, 'Spend_TV': spend_tv, 'Spend_Digital': spend_digital
})

def apply_geometric_adstock(spend_series, decay_rate):
    adstocked = np.zeros_like(spend_series)
    for t in range(len(spend_series)):
        if t == 0:
            adstocked[t] = spend_series[t]
        else:
            adstocked[t] = spend_series[t] + decay_rate * adstocked[t-1]
    return adstocked

# True latent target variable simulation
raw_adstock_tv = apply_geometric_adstock(df['Spend_TV'].values, decay_rate=0.60)
raw_adstock_dig = apply_geometric_adstock(df['Spend_Digital'].values, decay_rate=0.15)
df['Sales'] = (df['Base_Trend'] + df['Seasonality'] - (8.0 * df['Price']) +
               (0.45 * raw_adstock_tv) + (0.75 * raw_adstock_dig) +
               np.random.normal(0, 5, n_weeks))

# 2. Sensitivity Analysis Pipeline Implementation
# We test variations in TV's decay rate to observe impact on model metrics
decay_scenarios = [0.40, 0.50, 0.60, 0.70]
sensitivity_results = []

print("--- HYPERPARAMETER DECAY SENSITIVITY TESTING ---")
for lambda_val in decay_scenarios:
    X_matrix = pd.DataFrame({
        'Base_Trend': df['Base_Trend'],
        'Seasonality': df['Seasonality'],
        'Price': df['Price'],
        'Adstock_TV': apply_geometric_adstock(df['Spend_TV'].values, decay_rate=lambda_val),
        'Adstock_Digital': apply_geometric_adstock(df['Spend_Digital'].values, decay_rate=0.15)
    })

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_matrix)

    model = Ridge(alpha=10.0)
    model.fit(X_scaled, df['Sales'].values)

    # Extract unscaled coefficients
    unscaled_coefs = model.coef_ / scaler.scale_
    tv_coef = unscaled_coefs[3]
    dig_coef = unscaled_coefs[4]

    # Calculate R-squared for evaluation
    r2_score = model.score(X_scaled, df['Sales'].values)

    print(f"Scenario λ={lambda_val:.2f} | R²: {r2_score:.4f} | TV Coef: {tv_coef:.4f} | Digital Coef: {dig_coef:.4f}")


--- HYPERPARAMETER DECAY SENSITIVITY TESTING ---
Scenario λ=0.40 | R²: 0.9565 | TV Coef: 0.4693 | Digital Coef: 0.7353
Scenario λ=0.50 | R²: 0.9656 | TV Coef: 0.4583 | Digital Coef: 0.7312
Scenario λ=0.60 | R²: 0.9684 | TV Coef: 0.4301 | Digital Coef: 0.7274
Scenario λ=0.70 | R²: 0.9596 | TV Coef: 0.3787 | Digital Coef: 0.7239
